# Hands-on Exercise 1 — Naive vs. Multi-Stage Image Size Comparison
### AI Operations (AIOps) — Module 3, Lecture 1 | ~45 minutes

**Referenced in:** *Module3_Slides_1_Docker.pptx*, Hands-on Exercise 1

**Objective:** containerize a small training script two ways — a naive single-stage Dockerfile
and a multi-stage Dockerfile — and measure the concrete size difference. Add a `.dockerignore`
file, and reason through GPU passthrough even if no GPU hardware is available.

**Steps (from the slide deck):**
1. Write a naive, single-stage Dockerfile for the provided training script and build it; record the image size.
2. Rewrite it as a multi-stage build and build it; record the new image size and the percentage reduction.
3. Add a `.dockerignore` file and confirm it excludes unnecessary files from the build context.
4. Run the resulting container, with and without a `--gpus` flag, and note what changes (conceptually, if no GPU hardware is available).

**Deliverable:** a documented naive-vs-multi-stage image size comparison, a working `.dockerignore`,
and a clear explanation of what `--gpus all` would change on GPU-equipped hardware.

> **Prerequisites:** Docker Desktop or Docker Engine installed and running — verify with
> `docker run hello-world` before starting. This notebook checks for Docker automatically and
> prints clear instructions if it isn't available in your current environment.

## Step 0 — Check your environment

In [ ]:
import shutil, subprocess, os

DOCKER_AVAILABLE = shutil.which("docker") is not None
if DOCKER_AVAILABLE:
    result = subprocess.run(["docker", "info"], capture_output=True, text=True)
    DOCKER_AVAILABLE = result.returncode == 0

print("Docker CLI found on PATH:", shutil.which("docker") is not None)
print("Docker daemon reachable:", DOCKER_AVAILABLE)
if not DOCKER_AVAILABLE:
    print(
        "\nDocker is not available in this environment. The cells below will still write "
        "all the files you need; the docker build/run cells will print instructions instead "
        "of executing. Run this same notebook on a machine with Docker installed to complete "
        "the exercise for real."
    )

## Step 1 — Create a small training script and requirements file

In [ ]:
os.makedirs("docker_lab", exist_ok=True)

with open("docker_lab/train.py", "w") as f:
    f.write('''
print("Training a tiny model...")
import numpy as np
X = np.random.rand(1000, 10)
y = np.random.rand(1000)
w = np.linalg.lstsq(X, y, rcond=None)[0]
print("Learned weights:", w[:3], "...")
print("Done.")
''')

with open("docker_lab/requirements.txt", "w") as f:
    f.write("numpy\n")

print("Wrote docker_lab/train.py and docker_lab/requirements.txt")

## Step 2 — Write a NAIVE, single-stage Dockerfile

In [ ]:
naive_dockerfile = '''FROM python:3.10
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD ["python", "train.py"]
'''

with open("docker_lab/Dockerfile.naive", "w") as f:
    f.write(naive_dockerfile)

print(naive_dockerfile)

## Step 3 — Build the naive image and record its size

In [ ]:
if DOCKER_AVAILABLE:
    subprocess.run(
        ["docker", "build", "-t", "naive-image", "-f", "Dockerfile.naive", "."],
        cwd="docker_lab", check=True,
    )
    result = subprocess.run(
        ["docker", "images", "naive-image", "--format", "{{.Repository}}: {{.Size}}"],
        capture_output=True, text=True,
    )
    print(result.stdout)
else:
    print("$ cd docker_lab && docker build -t naive-image -f Dockerfile.naive .")
    print("$ docker images naive-image")
    print("\n(Run these commands yourself on a Docker-enabled machine, then record the size below.)")

_Your naive image size:_ 

**naive-image size:** ___ MB

## Step 4 — Rewrite as a MULTI-STAGE Dockerfile

In [ ]:
multistage_dockerfile = '''# --- Stage 1: build ---
FROM python:3.10 AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/build/deps -r requirements.txt

# --- Stage 2: minimal runtime ---
FROM python:3.10-slim
WORKDIR /app
COPY --from=builder /build/deps /usr/local/lib/python3.10/site-packages
COPY . .
CMD ["python", "train.py"]
'''

with open("docker_lab/Dockerfile.multistage", "w") as f:
    f.write(multistage_dockerfile)

print(multistage_dockerfile)

## Step 5 — Build the multi-stage image and compare sizes

In [ ]:
if DOCKER_AVAILABLE:
    subprocess.run(
        ["docker", "build", "-t", "multistage-image", "-f", "Dockerfile.multistage", "."],
        cwd="docker_lab", check=True,
    )
    result = subprocess.run(
        ["docker", "images", "multistage-image", "--format", "{{.Repository}}: {{.Size}}"],
        capture_output=True, text=True,
    )
    print(result.stdout)
else:
    print("$ cd docker_lab && docker build -t multistage-image -f Dockerfile.multistage .")
    print("$ docker images multistage-image")
    print("\n(Run these commands yourself, then record the size below.)")

_Your comparison:_

| Image | Size |
|---|---|
| naive-image | ___ MB |
| multistage-image | ___ MB |

**Percentage reduction:** ___%

**Why is it smaller?** (name what was left out of the final stage — be specific: which base image, which build artifacts)


## Step 6 — Add a `.dockerignore`

In [ ]:
dockerignore_content = '''.git
__pycache__
*.pyc
.venv
data/
*.ipynb_checkpoints
'''

with open("docker_lab/.dockerignore", "w") as f:
    f.write(dockerignore_content)

print(dockerignore_content)
print("Rebuild either image after adding this file -- the build context sent to the Docker")
print("daemon should now exclude these paths, which you can verify with the build output's")
print("'Sending build context to Docker daemon  X.XXMB' line at the start of the build log.")

## Step 7 — GPU passthrough (conceptual if no GPU hardware)

Even without a GPU-equipped machine, write out the exact commands you would run, and answer
the questions below.

In [ ]:
gpu_dockerfile = '''FROM nvidia/cuda:12.1.0-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y python3 python3-pip
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["python3", "train.py"]
'''
print(gpu_dockerfile)

print("Run WITHOUT GPU access:")
print("  $ docker run multistage-image python3 -c \"import torch; print(torch.cuda.is_available())\"")
print("  Expected: False\n")

print("Run WITH GPU access:")
print("  $ docker run --gpus all multistage-image python3 -c \"import torch; print(torch.cuda.is_available())\"")
print("  Expected: True (only on a host with an NVIDIA GPU + NVIDIA Container Toolkit installed)")

_Your answers:_

1. What flag grants a container access to ALL host GPUs? 
2. What flag restricts a container to exactly ONE specific GPU (e.g. GPU index 0)? 
3. Why must the CUDA version in the base image be *compatible with*, not *identical to*, the host driver's CUDA version?


## ✅ Deliverable Checklist
- [ ] Naive single-stage Dockerfile built, with recorded image size
- [ ] Multi-stage Dockerfile built, with recorded image size and % reduction
- [ ] A specific (not generic) explanation of what was excluded from the final stage
- [ ] A working `.dockerignore` file
- [ ] Correct `--gpus` flag syntax for both "all GPUs" and "one specific GPU" scenarios

*Next: proceed to `Lecture2_Kubernetes_KServe_Exercise.ipynb`.*